# 🎙️ AdriyTTS · Suara Kloning Adriy & Amy

**Cara pakai dari HP:**
1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Tekan ▶ pada **Langkah 1** dan tunggu sampai muncul *"✅ Siap"* (±5–8 menit, sekali per sesi).
3. Di **Langkah 2**, tulis teks dan pilih suara, lalu tekan ▶. File MP3 akan terunduh ke HP.
4. Ulangi Langkah 2 untuk teks lain (tidak perlu Langkah 1 lagi selama halaman masih terbuka).

*Gratis dari Google. GPU gratis kadang tidak tersedia; coba lagi beberapa saat kemudian.*

In [ ]:
#@title Langkah 1 · Siapkan mesin suara (sekali per sesi)
import os, subprocess, base64, json, sys
VOICES = json.loads('{"adriy": {"wav": "...", "text": "Bayangkan, bagaimana jika agama Islam pertama kali muncul bukan di jazirah Arab, melainkan di benua Eropa? Tepatnya di sebuah wilayah yang diselimuti salju, dengan pegunungan es, hutan lebat, dan musim dingin yang begitu ekstrem."}, "amy": {"wav": "...", "text": "Bayangkan kamu berjalan mengelilingi sebuah kursi, dan wajahmu harus terus menghadap kursi itu. Apa yang terjadi? Tanpa sadar, kamu ikut berputar satu kali penuh selama satu putaran. Nah, itulah yang dilakukan bulan."}}')
os.makedirs("/content/voices", exist_ok=True)
for n, v in VOICES.items():
    open(f"/content/voices/{n}.wav", "wb").write(base64.b64decode(v["wav"]))
print("Memasang VoxCPM2…")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "voxcpm", "num2words"], check=True)
import torch
dev = "GPU ✅" if torch.cuda.is_available() else "CPU (lambat — pilih T4 GPU di menu Runtime)"
print("Perangkat:", dev)
from voxcpm import VoxCPM
MODEL = VoxCPM.from_pretrained(os.environ.get("VOXCPM_MODEL", "openbmb/VoxCPM2"), load_denoiser=False)
print("✅ Siap. Lanjut ke Langkah 2.")

In [ ]:
#@title Langkah 2 · Tulis teks lalu tekan ▶
teks = "Halo semuanya, apa kabar? Hari ini kita akan membahas sejarah Kerajaan Majapahit." #@param {type:"string"}
suara = "adriy" #@param ["adriy", "amy"]
nama_file = "audio_adriytts" #@param {type:"string"}

import re, numpy as np, soundfile as sf, subprocess, os
from num2words import num2words

HURUF = dict(A="a",B="be",C="ce",D="de",E="e",F="ef",G="ge",H="ha",I="i",J="je",K="ka",L="el",M="em",N="en",
             O="o",P="pe",Q="ki",R="er",S="es",T="te",U="u",V="fe",W="we",X="eks",Y="ye",Z="zet")

def rapikan(t):
    t = re.sub(r"Rp\.?\s?(\d[\d.]*)", lambda m: m.group(1).replace(".", "") + " rupiah", t)
    t = re.sub(r"(\d+),(\d+)", lambda m: num2words(int(m.group(1)), lang="id") + " koma " + " ".join(num2words(int(d), lang="id") for d in m.group(2)), t)
    t = re.sub(r"\d{1,3}(?:\.\d{3})+", lambda m: m.group(0).replace(".", ""), t)
    t = re.sub(r"(\d+)%", lambda m: m.group(1) + " persen", t)
    t = re.sub(r"\d+", lambda m: num2words(int(m.group(0)), lang="id"), t)
    t = re.sub(r"(?<![A-Za-z])([A-Z]{1,4})(?![A-Za-z])", lambda m: " ".join(HURUF[c] for c in m.group(1)), t)
    return re.sub(r"\s+", " ", t).strip()

def potong(t, maks=260):
    kal = re.split(r"(?<=[.!?…])\s+", t)
    out, cur = [], ""
    for k in kal:
        if cur and len(cur) + len(k) > maks:
            out.append(cur); cur = k
        else:
            cur = (cur + " " + k).strip()
    if cur: out.append(cur)
    return out

ref = f"/content/voices/{suara}.wav"
bagian = potong(rapikan(teks))
sr = MODEL.tts_model.sample_rate
jeda = np.zeros(int(sr * 0.35), dtype=np.float32)
hasil = []
for i, b in enumerate(bagian, 1):
    print(f"Membuat bagian {i}/{len(bagian)}…")
    w = MODEL.generate(text=b, prompt_wav_path=ref, prompt_text=VOICES[suara]["text"],
                       reference_wav_path=ref, cfg_value=2.0, inference_timesteps=10)
    hasil += [np.asarray(w, dtype=np.float32), jeda]
sf.write("/content/_tmp.wav", np.concatenate(hasil), sr)
mp3 = f"/content/{nama_file}.mp3"
subprocess.run(["ffmpeg", "-v", "error", "-y", "-i", "/content/_tmp.wav", "-af", "loudnorm=I=-19:TP=-1.5",
                "-ac", "1", "-b:a", "128k", mp3], check=True)
print("✅ Selesai:", mp3)
from IPython.display import Audio, display
display(Audio(mp3))
try:
    from google.colab import files
    files.download(mp3)
except Exception:
    pass